In [1]:
import os
import torch
import torch.nn as nn
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from transformers import CLIPVisionModel
from google.colab import drive

### **1. LOAD MODEL INTO RAM**

In [2]:
# --- 1. CONFIGURATION ---
drive.mount('/content/drive')

# Update these paths to your actual locations
MODEL_WEIGHTS  = "/content/drive/MyDrive/Model/clip_classification_v5/best_model.pth"
MODEL_BASE     = "/content/drive/MyDrive/Model/clip_model" # HuggingFace base model path

DEVICE      = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
BATCH_SIZE  = 64

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [6]:
# --- 2. MODEL ARCHITECTURE ---
class ClassificationCLIP(nn.Module):
    def __init__(self, model_path):
        super().__init__()
        self.vision_encoder = CLIPVisionModel.from_pretrained(model_path)
        hidden_size = self.vision_encoder.config.hidden_size
        self.classifier = nn.Linear(hidden_size, 1)

    def forward(self, pixel_values):
        outputs = self.vision_encoder(pixel_values=pixel_values)
        return self.classifier(outputs.pooler_output)

In [7]:
# 3. Model Architecture
class ClassificationCLIP(nn.Module):
    def __init__(self, model_path):
        super().__init__()
        self.vision_encoder = CLIPVisionModel.from_pretrained(model_path)
        hidden_size = self.vision_encoder.config.hidden_size
        self.classifier = nn.Linear(hidden_size, 1)

    def forward(self, pixel_values):
        outputs = self.vision_encoder(pixel_values=pixel_values)
        return self.classifier(outputs.pooler_output)

In [5]:
# 4. Image Preprocessing
clip_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.48145466, 0.4578275, 0.40821073],
        std=[0.26862954, 0.26130258, 0.27577711]
    )
])

# 5. Initialize and Load Weights
print("Initializing Model...")
model = ClassificationCLIP(MODEL_BASE).to(DEVICE)
model.load_state_dict(torch.load(MODEL_WEIGHTS, map_location=DEVICE))
model.eval()
print(f"Model loaded successfully on {DEVICE}")

Initializing Model...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

CLIPVisionModel LOAD REPORT from: /content/drive/MyDrive/Model/clip_model
Key                                                          | Status     |  | 
-------------------------------------------------------------+------------+--+-
text_model.encoder.layers.{0...11}.self_attn.k_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.v_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm2.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.out_proj.bias   | UNEXPECTED |  | 
text_model.final_layer_norm.weight                           | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc1.bias              | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm2.weight        | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.out_proj.weight | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm1.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.q

Model loaded successfully on cuda


### **2. RUN EVALUATION**

In [9]:
import os
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from tqdm import tqdm

# --- SETTINGS ---
target_csv_path = '/content/drive/MyDrive/final_code_submission/5. Evaluation/output_impainting_sdxl_prnu.csv'
# Using a power of 2 for batch size (128) is generally more efficient for CUDA memory
BATCH_SIZE = 128
# Determine the number of CPU cores for parallel data loading
NUM_WORKERS = os.cpu_count()
# ----------------

# 1. Load the original file
print(f"Opening file: {target_csv_path}")
df = pd.read_csv(target_csv_path)

if 'path' not in df.columns:
    print("Error: Column 'path' not found. Please check your CSV.")
else:
    # 2. Setup Dataset & Loader
    class InferenceDataset(Dataset):
        def __init__(self, paths, transform):
            self.paths = paths
            self.transform = transform

        def __len__(self):
            return len(self.paths)

        def __getitem__(self, idx):
            path = self.paths[idx]
            try:
                # Basic check to avoid crashing if a file is missing
                if not os.path.exists(path):
                    return torch.zeros(3, 224, 224)
                img = Image.open(path).convert('RGB')
                return self.transform(img)
            except Exception:
                return torch.zeros(3, 224, 224)

    # Optimized DataLoader: Fixes Bottlenecks 2 & 3
    loader = DataLoader(
        InferenceDataset(df['path'].tolist(), clip_transform),
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,    # Parallelize image loading
        pin_memory=True,            # Lock RAM to speed up transfer to GPU
        prefetch_factor=2,          # Keep CPU ahead of GPU
        persistent_workers=True     # Save overhead by keeping workers alive
    )

    # 3. Run Inference
    preds, scores = [], []
    print(f"Evaluating {len(df)} images using {NUM_WORKERS} CPU workers...")

    # Put model in eval mode
    model.eval()

    with torch.no_grad():
        for imgs in tqdm(loader, desc="Processing"):
            # Fix 3: non_blocking=True works with pin_memory for async transfer
            imgs = imgs.to(DEVICE, non_blocking=True)

            with torch.amp.autocast(device_type='cuda', dtype=torch.float16):
                logits = model(imgs)

            # 1 = Real, 0 = Fake
            probs = torch.sigmoid(logits).squeeze(1).cpu().float().numpy()

            for p in probs:
                label = 1 if p >= 0.5 else 0
                conf = p if label == 1 else (1 - p)
                preds.append(label)
                scores.append(round(float(conf), 4))

    # 4. Write back to the SAME file
    df['CLIP_label_prediction'] = preds
    df['CLIP_confident_score'] = scores

    # Overwrite the original CSV with proper encoding for Excel
    df.to_csv(target_csv_path, index=False, encoding='utf-8-sig')

    print(f"\nSuccess! Updated original file at: {target_csv_path}")
    print(f"Added columns: CLIP_label_prediction & CLIP_confident_score")

    # Show preview
    display(df.head())

Opening file: /content/drive/MyDrive/final_code_submission/5. Evaluation/output_impainting_sdxl_prnu.csv
Evaluating 3000 images using 2 CPU workers...


Processing: 100%|██████████| 24/24 [13:26<00:00, 33.62s/it]


Success! Updated original file at: /content/drive/MyDrive/final_code_submission/5. Evaluation/output_impainting_sdxl_prnu.csv
Added columns: CLIP_label_prediction & CLIP_confident_score


,file_name,path,CLIP_label_prediction,CLIP_confident_score
0,197_adm_7_out_SDXL_spoofed_by_D30_Huawei_Honor...,/content/drive/MyDrive/output_impainting_sdxl_...,0,0.9963
1,GLIDE_1000_200_03_308_glide_00020_out_SDXL_spo...,/content/drive/MyDrive/output_impainting_sdxl_...,0,0.5276
2,VQDM_1000_200_03_381_vqdm_00143_out_SDXL_spoof...,/content/drive/MyDrive/output_impainting_sdxl_...,0,0.9837
3,618_biggan_00074_out_SDXL_spoofed_by_D08_Samsu...,/content/drive/MyDrive/output_impainting_sdxl_...,0,0.9511
4,VQDM_1000_200_00_050_vqdm_00074_out_SDXL_spoof...,/content/drive/MyDrive/output_impainting_sdxl_...,0,0.9871
